## Installs

In [43]:
RUNINSTALLS = False

if RUNINSTALLS:
  !pip install openpyxl
  !pip install --upgrade google-cloud-aiplatform


## Notebook Setup

In [44]:
from IPython.display import HTML, display
import IPython

def set_css(arg=None):
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
IPython.core.getipython.get_ipython().events.register('pre_run_cell', set_css)

import logging
import sys
format_string = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
logger = logging.getLogger()
fhandler = logging.FileHandler(filename='notebook.log', mode='a')
formatter = logging.Formatter(format_string)
fhandler.setFormatter(formatter)
#logger.addHandler(fhandler)  #uncomment if you want a log file
logging.basicConfig(format=format_string,
                     level=logging.INFO, stream=sys.stdout)
logger.setLevel(logging.INFO)


## Imports

In [45]:
from io import BytesIO
import datetime
import yaml
import json
from pathlib import Path
import pandas as pd
import openpyxl
import vertexai
from google.cloud import storage
from vertexai.generative_models import GenerationConfig, GenerativeModel, Part
from vertexai.preview import caching

## Project setup

In [46]:
BUCKET_NAME = "uk-bh-experiments-argolis-us"
FOLDER_PATH = "subsea7/hseq_data/"
MODEL_NAME = "gemini-1.5-pro-001"

vertexai.init(project="uk-bh-experiments-argolis", location="us-central1")

In [47]:
def clean_nones(value):
    """
    Recursively remove all None values from dictionaries and lists, and returns
    the result as a new dictionary or list.
    """
    if isinstance(value, list):
        return [clean_nones(x) for x in value if x is not None]
    elif isinstance(value, dict):
        return {
            key: clean_nones(val)
            for key, val in value.items()
            if val is not None
        }
    else:
        return value

def get_description_column(df):
  column_names = list(df)
  for column_name in column_names:
      if "desc" in column_name.lower():
         return column_name
  #if none found, return the 4th column
  return column_names[3]


def get_excel_sheet( file_name, sheet_name=None, header=0, mandatory_columns=None):

  storage_client = storage.Client()
  bucket = storage_client.bucket(BUCKET_NAME)
  blob = bucket.blob(file_name)
  logging.debug(f"Got file {file_name}")

  with blob.open("rb") as f:
      file_bytes = BytesIO(f.read())

  logging.info(f"read file {file_name}")

  openpyxl.reader.excel.warnings.simplefilter(action='ignore')

  if sheet_name is None:
    sheet_names = pd.ExcelFile(file_bytes,  engine='openpyxl').sheet_names
    print(f"Available sheets:")
    for sheet_name  in sheet_names:
        print(f"{sheet_name}")
    return sheet_names
  else:
    print(f"Reading sheet {sheet_name}")

    with pd.ExcelFile(file_bytes,  engine='openpyxl') as xls:
      df = pd.read_excel(xls, sheet_name, header=header)

      print(f"Sheet {sheet_name} has {df.shape[0]} rows and {df.shape[1]} columns")
      if mandatory_columns == None:
         mandatory_columns = [get_description_column(df)]
      logging.debug(f"Dropping all rows that have nothing in the columns: {mandatory_columns}")
      df.dropna(subset=mandatory_columns, inplace=True)
      #print(f"Cleaned Rows {sheet_name} has {df.shape[0]} rows and {df.shape[1]} columns")
      logging.debug("Dropping columns that have have nothing in any rows")
      df.dropna(axis=1, how="all", inplace = True)

      print(f"Cleaned {sheet_name} has {df.shape[0]} rows and {df.shape[1]} columns")
      print(f'Column names found are: {list(df.columns.values)}')

    return df


def df_to_json(df, json_name=""):
  sheet_json_str = df.to_json(orient='records')
  #print(f'{sheet_json_str[:1]}')
  sheet_json = json.loads(sheet_json_str)
  sheet_json = clean_nones(sheet_json)
  return sheet_json


def df_select_columns(df):
  print(df.head())
  print(f"{list(df)}")
  print(df.isnull().sum())


def excel2json_str(file_name, sheet_name=None, header=0, mandatory_columns=None):
  df = get_excel_sheet(file_name, sheet_name, header, mandatory_columns)
  df_json = {}
  df_json['file'] = Path(file_name).stem
  df_json['sheet'] = sheet_name
  df_json['content'] = df_to_json(df) 
  return df_json
  #return json.dumps(df_json)


def excel2yaml(file_name: str, sheet_name=None, header=0, mandatory_columns=None):
  df = get_excel_sheet(file_name, sheet_name, header, mandatory_columns)
  df_yaml = yaml.dump(df.to_dict(orient='records'),default_flow_style=None)
  return df_yaml


def upload_blob(bucket_name, file_contents, destination_blob_name):
  """Uploads a file to the bucket."""
  storage_client = storage.Client()
  bucket = storage_client.get_bucket(bucket_name)
  blob = bucket.blob(destination_blob_name)

  blob.upload_from_string(file_contents)

  print(f'File uploaded to {destination_blob_name}')

def save_excel_json(doc, folder_path = ""):
  #doc_json = json.load(doc)
  file_contents = json.dumps(doc['content'])
  file_name = f"{doc['file']}.{doc['sheet']}"
  upload_blob(BUCKET_NAME, file_contents, folder_path + file_name + ".json")

## Add Caching



In [48]:
def cache_document(documents):

    system_instruction = """
    You are an safety expert. You always stick to the facts in the sources provided, and never make up new facts.
    Now look at these lists of saftey observations, and answer the following questions.
    """

    contents = [Part.from_text(json.dumps(document)) for document in documents]

    cached_content = caching.CachedContent.create(
        model_name=MODEL_NAME,
        system_instruction=system_instruction,
        contents=contents,
        ttl=datetime.timedelta(minutes=60),
    )

    print(cached_content.name)
    return cached_content.name


def generate_from_cache(cache_id, question):


    cached_content = caching.CachedContent(cached_content_name=cache_id)

    model = GenerativeModel.from_cached_content(cached_content=cached_content)

    response = model.generate_content(question)

    return response.text


## Use JSON as TXT

In [49]:
def generate_using_text(documents, question):
    
    system_instruction = """
    You are an safety expert. You always stick to the facts in the sources provided, and never make up new facts.
    Now look at these lists of saftey observations, and answer the following questions.
    """

    prompt = f'''
        <task> Provide a helpful and factual answer to the question that the user has asked</task>
        <question>{question}</question>
        <output>As well as giving a summary answer to the question, provide 3-6 examples of obvervations from the documents that support your conclusion</output>
    '''

    model = GenerativeModel(MODEL_NAME)
    model = GenerativeModel(
    model_name=MODEL_NAME,
    system_instruction=[
        system_instruction,
    ],
)
    generation_config=GenerationConfig(
        temperature = 0.8
    )

    contents = [Part.from_text(json.dumps(document)) for document in documents]
    contents.append(prompt)
    response = model.count_tokens(contents)
    logging.debug(f"Prompt Token Count: {response.total_tokens}")
    logging.debug(f"Prompt Character Count: {response.total_billable_characters}")

    response = model.generate_content(contents,generation_config=generation_config,stream=False)

    # Response tokens count
    usage_metadata = response.usage_metadata
    logging.info(f"Response Prompt Token Count: {usage_metadata.prompt_token_count}")
    logging.info(f"Candidates Token Count: {usage_metadata.candidates_token_count}")
    logging.info(f"Total Token Count: {usage_metadata.total_token_count}")

    response_text = response.text

    return response_text


In [50]:
df1 = get_excel_sheet("subsea7/hseq_data/NormandSubsea.xlsx", "OBSERVATIONS", 1)
df1.head()
df_select_columns(df1)


2024-07-12 17:59:24,744 - root - INFO - read file subsea7/hseq_data/NormandSubsea.xlsx
Reading sheet OBSERVATIONS
Sheet OBSERVATIONS has 219 rows and 61 columns
Column names found are: ['Obs. No', 'Date', 'Name of observer', 'Department', 'Description of observation', 'Immediate Action', 'Further Action Required', 'Status', 'Observation close out comment/date input', 'Unnamed: 9', 'Eyes on path', 'Line of Fire', 'Use of tools & Equip', 'Pinch Points', '3-Point contact', 'Communication', 'Housekeeping', 'Pre Job Planning', 'Assistance need/used', 'Walking/working surfaces', 'Eyes on task', 'Hot Work preparation', 'Manual handling', 'Isolation systems', "use of Barriers & warning's", 'Conformance to rules', 'Reporting injury', 'PPE Head Protection', 'PPE Eye / Face Protection', 'PPE Respiratory ', 'PPE Protective Clothing', 'PPE Hand / Arm Protection', 'PPE Feet / Ankle Protection', 'PPE Hearing Protection', 'PPE WAH PPE', 'Unnamed: 35', 'Warning system inadequate', 'Defective tools /Equ

In [51]:
sheet_json_str = df2.to_json(orient='records')
observation = generate_using_text([sheet_json_str], "what is the most unsafe location")
print(f'{observation}')

I0000 00:00:1720803568.718416    7453 ev_epoll1_linux.cc:125] grpc epoll fd: 71


2024-07-12 17:59:36,939 - root - INFO - Response Prompt Token Count: 12507
2024-07-12 17:59:36,942 - root - INFO - Candidates Token Count: 192
2024-07-12 17:59:36,945 - root - INFO - Total Token Count: 12699
I cannot provide a definitive answer on the most unsafe location on the vessel based on the provided safety observations. 

While the observations highlight various safety concerns and their locations, they do not offer a quantifiable measure to compare and determine the absolute most unsafe location. 

For instance:

* **Observation 123** points to a potential slip hazard in the public toilet on the 1st deck due to a water leak. 
* **Observation 126** identifies a falling hazard in the online room where a fluorescent light bulb was hanging loose.
* **Observation 132** reports a similar falling hazard in the hangar with a loose light fixture and a corroded bolt on a damper. 

These observations suggest potential hazards in different areas, but without further information on the fre

I0000 00:00:1720803576.949990   10407 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720803576.952884   10407 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


In [52]:
json_docs = [
    excel2json_str("subsea7/hseq_data/NormandSubsea.xlsx", "OBSERVATIONS", 1)
]
question = "Where on our worksites is someone most likely to be hurt?  What were they doing?"
generate_using_text(json_docs, question)

2024-07-12 17:59:38,985 - root - INFO - read file subsea7/hseq_data/NormandSubsea.xlsx
Reading sheet OBSERVATIONS
Sheet OBSERVATIONS has 219 rows and 61 columns
Column names found are: ['Obs. No', 'Date', 'Name of observer', 'Department', 'Description of observation', 'Immediate Action', 'Further Action Required', 'Status', 'Observation close out comment/date input', 'Unnamed: 9', 'Eyes on path', 'Line of Fire', 'Use of tools & Equip', 'Pinch Points', '3-Point contact', 'Communication', 'Housekeeping', 'Pre Job Planning', 'Assistance need/used', 'Walking/working surfaces', 'Eyes on task', 'Hot Work preparation', 'Manual handling', 'Isolation systems', "use of Barriers & warning's", 'Conformance to rules', 'Reporting injury', 'PPE Head Protection', 'PPE Eye / Face Protection', 'PPE Respiratory ', 'PPE Protective Clothing', 'PPE Hand / Arm Protection', 'PPE Feet / Ankle Protection', 'PPE Hearing Protection', 'PPE WAH PPE', 'Unnamed: 35', 'Warning system inadequate', 'Defective tools /Equ

I0000 00:00:1720803582.354923    7453 ev_epoll1_linux.cc:125] grpc epoll fd: 72


2024-07-12 17:59:54,000 - root - INFO - Response Prompt Token Count: 5812
2024-07-12 17:59:54,003 - root - INFO - Candidates Token Count: 473
2024-07-12 17:59:54,010 - root - INFO - Total Token Count: 6285


I0000 00:00:1720803594.016528   10456 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720803594.022588   10456 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


"The safety observations in this document point towards a few key areas of concern:\n\n**1. Working at Height/Dropped Objects:** Several observations highlight risks related to falling objects and working at height. \n\n*   **Obs. No. 124:** A rescue dummy used for training had safety glasses on, posing a drop hazard. \n*   **Obs. No. 126:** A loose fluorescent light bulb posed a risk of falling and breaking. \n*   **Obs. No. 132:** A corroded bolt on a hangar light fixture could have caused the light to fall. \n* **Obs. No 151:** Folders stored above a desk had the potential to fall. \n\n**2. Housekeeping and Material Storage:** Issues related to poor housekeeping and inadequate storage of materials are evident.\n\n*   **Obs. No. 127:** Oil barrels were not stored on a bund or drip tray, risking a potential spill. \n*   **Obs. No. 139:** Missing seafastening bars in the galley's provisions stores and freezer could lead to items falling during rough seas.\n\n**3. Equipment/Material Iss

In [53]:
json_docs = [
    excel2json_str("subsea7/hseq_data/SevenArctic.xlsx", "OBS", 2),
    excel2json_str("subsea7/hseq_data/NormandSubsea.xlsx", "OBSERVATIONS", 1),
    excel2json_str(f"{FOLDER_PATH}SeawayStrashnov.xlsx", "Obs_Int Register", 1),
    excel2json_str(f"{FOLDER_PATH}SeawayMoxie.xlsx", "Obs_Int Register", 10),
    excel2json_str(f"{FOLDER_PATH}SevenOceans.xlsx", "Safety SO record", 59),
    excel2json_str(f"{FOLDER_PATH}SevenPacific_Observations.xlsx", "CSB 2021", 1),
    excel2json_str(f"{FOLDER_PATH}SevenVega.xlsx", "SEVEN VEGA CSBs", 0),

    ]    

2024-07-12 17:59:57,088 - root - INFO - read file subsea7/hseq_data/SevenArctic.xlsx
Reading sheet OBS
Sheet OBS has 7538 rows and 19 columns
Column names found are: ['Card\nID', 'OBS', 'Day', 'Month', 'Year', 'Time', 'Location', 'Name of Observer', 'Department', 'Description', 'Corrective/Immediate Action Taken', 'Suggested Further Action Required', 'Person Responsible\nfor Action', 'Action Undertaken', 'OPEN / Closed', 'Date\nClosed DD/MM/YYYY', 'Safety Improvement (Yes/No)', 'Unnamed: 17', 'Unnamed: 18']
Cleaned OBS has 7507 rows and 19 columns
2024-07-12 18:00:40,717 - root - INFO - read file subsea7/hseq_data/NormandSubsea.xlsx
Reading sheet OBSERVATIONS
Sheet OBSERVATIONS has 219 rows and 61 columns
Column names found are: ['Obs. No', 'Date', 'Name of observer', 'Department', 'Description of observation', 'Immediate Action', 'Further Action Required', 'Status', 'Observation close out comment/date input', 'Unnamed: 9', 'Eyes on path', 'Line of Fire', 'Use of tools & Equip', 'Pinch

In [54]:
for doc in json_docs:
    save_excel_json( doc, FOLDER_PATH)

File uploaded string to subsea7/hseq_data/SevenArctic.OBS.json
File uploaded string to subsea7/hseq_data/NormandSubsea.OBSERVATIONS.json
File uploaded string to subsea7/hseq_data/SeawayStrashnov.Obs_Int Register.json
File uploaded string to subsea7/hseq_data/SeawayMoxie.Obs_Int Register.json
File uploaded string to subsea7/hseq_data/SevenOceans.Safety SO record.json
File uploaded string to subsea7/hseq_data/SevenPacific_Observations.CSB 2021.json
File uploaded string to subsea7/hseq_data/SevenPacific_Observations.CSB 2021.json


## Questions

Questions from John:
 - Where on our worksites is someone most likely to be hurt?  What were they doing?
 - What is the most likely way someone could be hurt?
 - Do we have a problem with doors?
 - If I was to tackle one issue on our worksites, what would it be?
 - Are our gallies safe?
 

Can AI be used on our Synergi, RA7 and MOC databases?
(Synergi is our HSEQ incident reporting tool, RA7 records risk assessments, MOC is our Management of Change tool)

 - When we change rigging offshore, do we normally increase or decrease the capacity?
 - Do we have a problem with dropped tools?

In [55]:
question = "Where on our worksites is someone most likely to be hurt?  What were they doing?"
generate_using_text(json_docs, question)


I0000 00:00:1720803759.521787    7453 ev_epoll1_linux.cc:125] grpc epoll fd: 97
I0000 00:00:1720803759.860165   10890 subchannel.cc:806] subchannel 0x7f63d000f7d0 {address=ipv6:%5B2a00:1450:4009:823::200a%5D:443, args={grpc.client_channel_factory=0x8013ad0, grpc.default_authority=us-central1-aiplatform.googleapis.com:443, grpc.dns_enable_srv_queries=1, grpc.http2_scheme=https, grpc.internal.channel_credentials=0x8666aa0, grpc.internal.client_channel_call_destination=0x7f640de4b390, grpc.internal.event_engine=0x7f63d002a8b0, grpc.internal.security_connector=0x7f63d0001370, grpc.internal.subchannel_pool=0x8013460, grpc.max_receive_message_length=-1, grpc.max_send_message_length=-1, grpc.primary_user_agent=grpc-python/1.65.0, grpc.resource_quota=0x8016190, grpc.server_uri=dns:///us-central1-aiplatform.googleapis.com:443}}: connect failed (UNKNOWN:connect: Network is unreachable (101) {created_time:"2024-07-12T18:02:39.857308148+01:00"}), backing off for 1000 ms


2024-07-12 18:04:39,702 - root - INFO - Response Prompt Token Count: 1656829
2024-07-12 18:04:39,707 - root - INFO - Candidates Token Count: 249
2024-07-12 18:04:39,708 - root - INFO - Total Token Count: 1657078


I0000 00:00:1720803879.718451   10896 tcp_posix.cc:809] IOMGR endpoint shutdown


'The most likely place for someone to be hurt on the worksites is **the main deck, particularly around the cranes**, due to **poor housekeeping and dropped objects.**\n\nHere are some supporting observations:\n\n* **SevenArctic, Obs ID 34:** "Loose objects on tower with potential to fall."\n* **SevenArctic, Obs ID 36:** "Long reach paint brush left tied to tower. Potential to fall." \n* **SevenArctic, Obs ID 168:** "Carton box left behind, with different materials and tools, inside hydraulic machinery space."\n* **SevenArctic, Obs ID 253:** "Electrical cable (approx 30m length) left abandoned at top of stairway. Trip hazard."\n* **SevenArctic, Obs ID 427:** "220t a&R winch block found with no locking pin. Locking pin found nearby. Locking pin was applied but wasn\'t split." \n\nThese observations highlight recurring issues with tools and equipment being left in unsafe positions, creating hazards for personnel working on deck. This emphasizes the need for consistent good housekeeping pr

I0000 00:00:1720803879.735525   10896 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


In [56]:
question = "What is the most likely way someone could be hurt?"
observation = generate_using_text(json_docs, question)
print(f'{observation}')

I0000 00:00:1720803879.979219    7453 ev_epoll1_linux.cc:125] grpc epoll fd: 90


2024-07-12 18:05:23,043 - root - INFO - Response Prompt Token Count: 1656821
2024-07-12 18:05:23,047 - root - INFO - Candidates Token Count: 158
2024-07-12 18:05:23,050 - root - INFO - Total Token Count: 1656979
The most likely way someone could be hurt on the Seven Arctic is by tripping over an object. There are numerous safety observations relating to trip hazards, for example:

*  **Observation 2:** "Found tap in bridge cleaning locker not properly closed and dripping. Waste of water. "
*  **Observation 10:** "Loose grating on deck of messroom to left of hot counter"
*  **Observation 33:** "Wet stairwell, slip hazard"
*  **Observation 53:** "Cable ties being used to colour code loose rigging- can be easily removed/ damaged"
*  **Observation 116:** "Crew tripping on messroom chair plate. Suggestion to remove plate until chair can be re-attached" 



I0000 00:00:1720803923.059465   10970 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720803923.062229   10970 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


In [57]:
question = "Do we have a problem with doors?"
observation = generate_using_text(json_docs, question)
print(f'{observation}')

I0000 00:00:1720803923.247898    7453 ev_epoll1_linux.cc:125] grpc epoll fd: 90


2024-07-12 18:06:08,753 - root - INFO - Response Prompt Token Count: 1656818
2024-07-12 18:06:08,757 - root - INFO - Candidates Token Count: 189
2024-07-12 18:06:08,759 - root - INFO - Total Token Count: 1657007
Yes, there are numerous observations about doors. Many observations note doors being left open, unsecured, or malfunctioning. 

Here are some examples from the SevenArctic observations log:

* **Observation 3:**  "Found tap in bridge cleaning locker not properly closed and dripping. Waste of water."
* **Observation 24:** "During fire drill some double fire doors closed such that the inner door was outside the outer door and there wasn’t a proper seal."
* **Observation 29:** "Lots of noise in accomodation for offshift personnel trying to sleep"
* **Observation 48:** "Crane gate insecure and banging"
* **Observation 68:** "Cabin 408 wont open"
* **Observation 165:** "WTD 10 left open" 

These examples demonstrate that doors on the SevenArctic have been a recurring issue. 



I0000 00:00:1720803968.768537   10991 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720803968.770818   10991 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


In [58]:
question = "If I was to tackle one issue on our worksites, what would it be?"
observation = generate_using_text(json_docs, question)
print(f'{observation}')

I0000 00:00:1720803968.973853    7453 ev_epoll1_linux.cc:125] grpc epoll fd: 90


2024-07-12 18:06:52,572 - root - INFO - Response Prompt Token Count: 1656827
2024-07-12 18:06:52,573 - root - INFO - Candidates Token Count: 224
2024-07-12 18:06:52,575 - root - INFO - Total Token Count: 1657051
The safety logs highlight a recurring and serious issue with **weathertight doors (WTDs) being left open and unsecured**. This poses a significant risk to personnel safety and vessel integrity. 

Here are some examples from the safety observations:

* **SevenArctic, Entry #209:** "WTD entrance to Bow from stairway found open fully  & not secured"
* **SevenArctic, Entry #607:** "2x weathertight doors open and not secured back"
* **SevenArctic, Entry #922:** "Watertight door 12 left open at sea"
* **SevenArctic, Entry #1612:** "Stbd main deck door left swinging again, despite TOFs talk at safety meeting."
* **SeawayStrashnov, Entry #117:** "If you walk through a closed WTD, please close it behind you. In daytime we are conducting Main Crane Load Test. WTD must remain closed."

Fo

I0000 00:00:1720804012.580268   11063 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720804012.581961   11063 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


## Try as YAML

In [63]:

df_yaml = yaml.dump(clean_nones(df1.to_dict(orient='records')),default_flow_style=None)

In [64]:
print(f'{df_yaml[:1000]}')

- {Certification Issue: .nan, Communication: .nan, Conformance to rules: .nan, Date: !!timestamp '2021-01-02
    00:00:00', Defective tools /Equipment: .nan, Department: Project - Deck, Description of observation: Suggestion
    to weld a padeye directly below MHS main lift.  This will eliminate trip hazards
    and give a direct seafastening point., Equipment/Material; issue: .nan, Eyes on path: .nan,
  Eyes on task: .nan, Further Action Required: 'This will be reviewed first by the
    Deck Fmn and OM before any actions are taken. ', Housekeeping: .nan, Immediate Action: Spoke
    with the welder to look into modifying grating to suit padeye., Improvement suggestion: S,
  Inadequate guard's and Barriers: .nan, Line of Fire: .nan, Name of observer: Brian
    Bullock, Objects with potential to fall: .nan, Obs. No: 120.0, Observation close out comment/date input: &id001 !!timestamp '2021-03-04
    00:00:00', PPE Eye / Face Protection: .nan, PPE Feet / Ankle Protection: .nan,
  PPE Hand 

In [65]:
observation = generate_using_text(df_yaml, "what is the most unsafe location")
print(observation)

I0000 00:00:1720804239.058711    7453 ev_epoll1_linux.cc:125] grpc epoll fd: 71


2024-07-12 18:12:25,404 - root - INFO - Response Prompt Token Count: 121732
2024-07-12 18:12:25,405 - root - INFO - Candidates Token Count: 137
2024-07-12 18:12:25,405 - root - INFO - Total Token Count: 121869
It is impossible to determine the most unsafe location from the provided safety observations. The observations describe various safety concerns and their resolutions in different departments without providing a quantifiable measure of unsafety for each location. 

For example:

* Observation 120 highlights a trip hazard on the deck, suggesting a padeye be welded for a direct sea fastening point.
* Observation 123 reports a warm water leak in the public toilet on the 1st deck, causing a potential slipping hazard.
* Observation 132 reports a corroded bolt on a vibration damper causing a light to hang down in the hangar, posing a risk of objects falling. 



I0000 00:00:1720804345.407254   11169 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720804345.429949   11169 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce
